# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The record sets, fields, and columns are referenced by their `@id`. This ensures consistent identification throughout reproducible workflows.

In [ ]:
# List all available record sets and their fields. We access them with their @id per Croissant schema.
print("Available Record Sets:\n----------------------")
for rset in metadata.record_sets:
    print(f"@id: {rset['@id']}")
    print(f"  name: {rset.get('name','')}\n  description: {rset.get('description','')}")
    print("  Fields:")
    for field in rset.get('field', []):
        field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
        print(f"    - {field_id}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we demonstrate how to extract all record sets and load them into pandas DataFrames, referencing by `@id`.

In [ ]:
# Gather all record set @ids
record_sets_ids = [rset['@id'] for rset in metadata.record_sets]
print(f"Record set @ids found: {record_sets_ids}\n")

dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading records from record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  {df.shape[0]} rows x {df.shape[1]} columns\n")

# Choose one primary record set for demonstration (first one, if available)
if len(record_sets_ids) > 0:
    primary_record_set_id = record_sets_ids[0]
    primary_df = dataframes[primary_record_set_id]
    print(f"Columns in {primary_record_set_id}:\n{list(primary_df.columns)}\n")
    display(primary_df.head())
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Below, we demonstrate EDA on a numeric field from the primary record set.

In [ ]:
# -- Example EDA for a numeric field --
# Please adjust <numeric_field_id> and <group_field_id> to match an actual @id from your selected record set fields.

primary_df = dataframes[primary_record_set_id]  # Already loaded

# Attempt to auto-detect a numeric field (float or int columns)
numeric_candidate = None
for col in primary_df.columns:
    if pd.api.types.is_numeric_dtype(primary_df[col]):
        numeric_candidate = col
        break

if numeric_candidate is not None:
    numeric_field_id = numeric_candidate
    print(f"Using numeric field: {numeric_field_id}\n")

    threshold = primary_df[numeric_field_id].mean() if pd.notnull(primary_df[numeric_field_id].mean()) else 0
    filtered_df = primary_df[primary_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    normed_col = f"{numeric_field_id}_normalized"
    filtered_df[normed_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, normed_col]].head())

    # Attempt to pick a non-numeric column for grouping
    group_field_id = None
    for col in primary_df.columns:
        if not pd.api.types.is_numeric_dtype(primary_df[col]) and pd.api.types.is_string_dtype(primary_df[col]):
            group_field_id = col
            break
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field detected for EDA. Please specify a numeric field @id.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field if detected
if numeric_candidate is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(primary_df[numeric_candidate].dropna(), kde=True, bins=30)
    plt.xlabel(numeric_candidate)
    plt.title(f"Distribution of {numeric_candidate} in {primary_record_set_id}")
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_candidate, data=primary_df)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"{numeric_candidate} by {group_field_id}")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded and inspected the mass spectrometry dataset using `mlcroissant`, exploring the available record sets and their fields.
- We demonstrated how to extract a primary record set, perform EDA, filter and normalize a numeric field, and visualize distributions.
- Record set, field, and column references were handled via their Croissant schema `@id` for clarity and reproducibility.

For further analysis, we recommend reviewing all available fields in the dataset and customizing filtering/grouping based on specific research needs.